# FLUX.1-dev LoRA — Islamic Parametric Architecture

Production training notebook. Hyperparameters (locked to spec):

| Hyperparameter | Value |
|---|---|
| Base model | `black-forest-labs/FLUX.1-dev` |
| LoRA rank / alpha | 16 / 16 |
| Learning rate | 1e-4 |
| Resolution | 1024×1024 |
| Max steps | 800 (800–1000 allowed) |
| Optimizer | adamw8bit |
| Mixed precision | bf16 + gradient checkpointing |
| Checkpointing | every 250 steps (`checkpoint-250/500/750/800`), resume via `--resume` |
| Trigger word | `in Islamic_Parametric style` |
| Output | `pytorch_lora_weights.safetensors` |

**Runtime:** Google Colab → Change runtime → GPU (T4 works slowly; A100/L4 recommended).
**Access:** FLUX.1-dev is gated — accept the license at huggingface.co/black-forest-labs/FLUX.1-dev and set `HF_TOKEN` below with a token that has access.

In [ ]:
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
!pip install -q "diffusers>=0.31.0" "transformers>=4.49.0" "accelerate>=0.31.0" "safetensors>=0.4.0" "huggingface_hub>=0.23" tensorboard

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
REPO_URL = 'https://github.com/Shehab-Hegab/flux-islamic-parametric'
if not Path('train_flux_lora.py').exists():
    try:
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, 'repo'],
            check=True,
            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
        )
        for item in Path('repo').iterdir():
            dest = Path(item.name)
            if not dest.exists():
                shutil.move(str(item), str(dest))
        shutil.rmtree('repo', ignore_errors=True)
        print('cloned public repo into cwd')
    except Exception as exc:
        print('git clone failed:', exc)
        print('will try Hugging Face dataset fallback next cell')
print('cwd ready:', os.getcwd(), 'train script:', Path('train_flux_lora.py').exists())

In [ ]:
from pathlib import Path
import os
ds = Path('dataset_islamic_parametric')
def _count(d: Path):
    imgs = sorted(d.glob('image_*.jpg')) if d.exists() else []
    caps = sorted((d / 'captions').glob('*.txt')) if (d / 'captions').exists() else []
    return imgs, caps
images, captions = _count(ds)
if len(images) == 0:
    print('local dataset missing — downloading from Hugging Face...')
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN') or None
    snapshot_download(
        repo_id='Shehab-Hegab/islamic-parametric-architecture-dataset',
        repo_type='dataset',
        local_dir=str(ds),
        token=token,
    )
    images, captions = _count(ds)
print(f'images: {len(images)} captions: {len(captions)}')
assert len(images) >= 1, 'dataset download failed — open HF dataset and upload folder manually'
print((ds / 'captions' / 'image_01.txt').read_text() if captions else 'no captions yet')

In [ ]:
!python generate_captions.py --backend template || true
!python train_flux_lora.py --fetch-script

## Training launch
Runs `accelerate launch training/train_dreambooth_lora_flux.py` via the wrapper. On a Colab T4 expect roughly 2–4 hours for 800 steps at 1024² with batch 1 + grad accum 4 + bf16 + gradient checkpointing. A100 is substantially faster.

Saves state checkpoints every 250 steps under `output/checkpoint-*`. If the session disconnects, re-run with `--resume` to continue from `checkpoint-800`/`latest`. After finishing, optionally `huggingface-cli upload` intermediate checkpoints to the LoRA repo.

In [ ]:
!python train_flux_lora.py --dry-run

In [ ]:
%env TOKENIZERS_PARALLELISM=false
!python train_flux_lora.py --instance-dir dataset_islamic_parametric --output-dir output

## Resume (after disconnect)
If Colab dropped mid-run, execute the next cell to continue from the latest `output/checkpoint-*`.

In [ ]:
from pathlib import Path
ckpts = sorted(Path('output').glob('checkpoint-*')) if Path('output').exists() else []
print('checkpoints:', [c.name for c in ckpts])
resume = bool(ckpts)
if resume:
    print('will resume from latest checkpoint')
else:
    print('no checkpoints yet - run the training cell first')
if resume:
    get_ipython().run_line_magic('env', 'TOKENIZERS_PARALLELISM=false')
    get_ipython().system('python train_flux_lora.py --instance-dir dataset_islamic_parametric --output-dir output --resume')

In [ ]:
from pathlib import Path
weights = Path('output/pytorch_lora_weights.safetensors')
print('weights:', weights, 'exists:', weights.exists(), 'size_mb:', weights.stat().st_size // 1024**2 if weights.exists() else 0)
if weights.exists():
    # optional: push intermediate + final weights to the gated LoRA repo
    # requires HF_TOKEN with write access, only after training completes
    pass

In [ ]:
!python inference_eval.py --check || true
!python inference_eval.py --lora-path output || true
!python evaluate_structure.py --check || true
!python evaluate_structure.py --images dataset_islamic_parametric --limit 5 || true

In [ ]:
!python upload_to_hf.py --dry-run